In [ ]:
import boto3
import io
import json
import pandas as pd

BUCKET = "livewell-data-prod"
TABLE = "livewell-signals-prod"
REGION = "us-east-1"

s3 = boto3.client("s3", region_name=REGION)
dynamo = boto3.resource("dynamodb", region_name=REGION)
table = dynamo.Table(TABLE)

In [ ]:
body = s3.get_object(Bucket=BUCKET, Key="backtest/summary.json")["Body"].read()
summary = json.loads(body)
trades = summary["trades"]
print(f"Loaded {len(trades)} trades from backtest summary")

win_map = {t["signal_id"]: t["win"] for t in trades}
signal_ids = list(win_map.keys())
print(f"Unique signal_ids: {len(signal_ids)}")

In [ ]:
def batch_get_signals(signal_ids: list[str]) -> list[dict]:
    records = []
    chunk_size = 25
    for i in range(0, len(signal_ids), chunk_size):
        chunk = signal_ids[i:i + chunk_size]
        response = boto3.client("dynamodb", region_name=REGION).batch_get_item(
            RequestItems={
                TABLE: {
                    "Keys": [{"signal_id": {"S": sid}} for sid in chunk],
                    "ProjectionExpression": "signal_id, s3_key, #d, ema_20, ema_50, rsi_14, macd, macd_signal, macd_hist, atr_14, session_quality, direction, signal_valid",
                    "ExpressionAttributeNames": {"#d": "date"},
                }
            }
        )
        for item in response["Responses"].get(TABLE, []):
            record = {k: list(v.values())[0] for k, v in item.items()}
            records.append(record)
        unprocessed = response.get("UnprocessedKeys", {})
        if unprocessed:
            print(f"Warning: unprocessed keys in chunk {i//chunk_size}")
    return records

raw_records = batch_get_signals(signal_ids)
print(f"Fetched {len(raw_records)} signal records from DynamoDB")

In [ ]:
_DIR_MAP = {"call": "buy", "put": "sell"}

rows = []
missing = 0
for rec in raw_records:
    sid = rec.get("signal_id")
    if sid not in win_map:
        missing += 1
        continue
    direction = rec.get("direction", "none")
    rows.append({
        "signal_id": sid,
        "s3_key": rec.get("s3_key", ""),
        "date": rec.get("date", ""),
        "ema_20": float(rec.get("ema_20", 0) or 0),
        "ema_50": float(rec.get("ema_50", 0) or 0),
        "rsi_14": float(rec.get("rsi_14", 50) or 50),
        "macd": float(rec.get("macd", 0) or 0),
        "macd_signal": float(rec.get("macd_signal", 0) or 0),
        "macd_hist": float(rec.get("macd_hist", 0) or 0),
        "atr_14": float(rec.get("atr_14", 0) or 0),
        "session_quality": rec.get("session_quality", "medium"),
        "direction": _DIR_MAP.get(direction, direction),
        "signal_valid": str(rec.get("signal_valid", "False")).lower() == "true",
        "label": int(bool(win_map[sid])),
    })

labeled_df = pd.DataFrame(rows)
print(f"Labeled rows: {len(labeled_df)}")
print(f"Missing from win_map: {missing}")
print(f"Label distribution:\n{labeled_df['label'].value_counts()}")

buf = io.BytesIO()
labeled_df.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="labeled/signals_labeled.parquet", Body=buf.read())
print(f"Written to s3://{BUCKET}/labeled/signals_labeled.parquet")

In [ ]:
verify_body = s3.get_object(Bucket=BUCKET, Key="labeled/signals_labeled.parquet")["Body"].read()
verify_df = pd.read_parquet(io.BytesIO(verify_body))
print(f"Verified: {len(verify_df)} rows, columns: {list(verify_df.columns)}")
print(f"Label=1: {verify_df['label'].sum()} ({verify_df['label'].mean():.1%})")
print(f"Label=0: {(verify_df['label'] == 0).sum()}")